# Getting Started with DelaDect

This notebook follows the Getting Started documentation. [for AI agent, include here the link]

## Path handling (hide me)

In [1]:
from pathlib import Path

from deladect.detection import crack_analysis, DelaminationDetector
from deladect.specimen import Specimen


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate repo root (missing {marker!r}) above {Path.cwd()}")


repo_root = find_repo_root()
data_root = repo_root / "example_images" / "sample-1"

## 1. Building the specimen class

The specimen class is the foundational dataclass of DelaDect. Before any analysis, a specimen with the necessary information must be created. For that, plies are added to the specimen class as shown below.

In [2]:
specimen = Specimen(
    name="sample-1-quickstart",
    scale_px_mm=41.03328366,
    path_full=str(data_root / "full"),
    path_upper_border=str(data_root / "upper"),
    path_middle=str(data_root / "middle"),
    path_lower_border=str(data_root / "lower"),
    image_types=["png"],
    results_root=str(repo_root / "results"),
    avg_crack_width_px=8.0,
)

specimen.add_ply(name="ply_0", orientation_deg=0.0)
specimen.add_ply(name="ply_90", orientation_deg=90.0)

interface = specimen.add_interface(name="i0", upper_ply_index=0, lower_ply_index=1)

[ply.name for ply in specimen.plies], interface.name

(['ply_0', 'ply_90'], 'i0')

## 2. Run crack detection

`crack_analysis` reads the specimen layup and returns a dict keyed by orientation (`"0"`, `"90"`), each holding its own `cracks` / `densities` / `thresholds` / `metrics` / `paths` / `params`.

In [3]:
crack_results = crack_analysis(
    specimen,
    export_images=True,
    background=True,
    save_cracks=True,
)

crack_results.keys()

dict_keys(['0', '90'])

## 3. Delamination detection

`DelaminationDetector` exposes edge and diffuse detection as two peer sub-detectors:

- `detector.edge.detect_primary(...)` — edge-only detection
- `detector.diffuse.diffuse_delamination(...)` — diffuse-only detection (crack-guided ROIs)
- `detector.detect_both_delaminations(...)` — combined edge + diffuse, with overlap arbitration

Shared infrastructure (preprocessing, caching) lives directly on `DelaminationDetector` and is reused by both sub-detectors.

In [4]:
detector = DelaminationDetector(specimen, interface, save_preprocess_outputs=True)

# The complete analysis result can be passed directly to diffuse detection.
cache_paths = detector.preprocess_stack_to_disk(
    specimen.image_stack_full,
    key="sample1_quickstart",
    reference_mode="rolling_median",
    reference_window=3,
    reference_skip=1,
)["cache_paths"]

len(cache_paths)

5

### 3a. Edge-only detection — `detector.edge.detect_primary`

In [5]:
edge_result = detector.edge.detect_primary(
    processed_cache_paths=cache_paths,
    save_overlays=False,
    debug=True,
)

edge_result.keys()

dict_keys(['masks', 'debug'])

### 3b. Diffuse-only detection — `detector.diffuse.diffuse_delamination`

In [6]:
diffuse_result = detector.diffuse.diffuse_delamination(
    cracks=crack_results,
    processed_cache_paths=cache_paths,
    save_overlays=False,
    debug=True,
)

diffuse_result.keys()

dict_keys(['masks', 'debug'])

### 3c. Combined edge + diffuse — `detect_both_delaminations`

This is the orchestrator: it runs edge detection via `self.edge`, diffuse crack-tracking via `self.diffuse`, and resolves overlaps in favour of edge damage.

In [7]:
result = detector.detect_both_delaminations(
    cracks=crack_results,
    avg_crack_width_px=8.0,
    save_overlays=True,
    overlay_view="classified",
    save_component_overlays=True,
    save_masks=True,
    save_metrics=True,
    edge_exclusion_px=5,
    progress=True,
)

result["paths"]

[progress] preprocess_stack: start (5 frames)


[progress] preprocess_stack: 25% (2/5)


[progress] preprocess_stack: 50% (3/5)


[progress] preprocess_stack: 75% (4/5)


[progress] preprocess_stack: 90% (5/5)


...

[progress] combined_delamination: 50% (3/5)


[progress] combined_delamination: 75% (4/5)


[progress] combined_delamination: 90% (5/5)


[progress] combined_delamination: done (5/5)


{'edge_raw_masks': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\\results\\sample-1-quickstart\\delamination\\both\\masks\\edge_raw.npz',
 'edge_exclusion_masks': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\\results\\sample-1-quickstart\\delamination\\both\\masks\\edge_exclusion.npz',
 'diffuse_raw_masks': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\\results\\sample-1-quickstart\\delamination\\both\\masks\\diffuse_raw.npz',
 'diffuse_masks': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\\results\\sample-1-quickstart\\delamination\\both\\masks\\diffuse_final.npz',
 'combined_masks': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\\results\\sample-1-quickstart\\delamination\\both\\masks\\combined.npz',
 'metrics': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\\results\\sample-1-quickstart\\delamination\\both\\metrics\\frame_metrics.csv',
 'combined_overlays': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\\results\\sample-1-quickstart\\delamination\\both\\ove

## 4. Save and reload the specimen

The specimen manifest records paths and configuration so a later session can reload cracks/delamination results without recomputation.

In [8]:
from deladect.io import load_specimen, save_specimen

manifest = specimen.results_dir("config") / "sample-1-quickstart.json"
save_specimen(specimen, manifest)

specimen_reloaded = load_specimen(
    manifest,
    auto_init_stacks=False,
    load_results=True,
    verbose=True,
)

specimen_reloaded.name

Found cracks for ply 'ply_0' (5 frames).
Found cracks for ply 'ply_90' (5 frames).
Found edge/diffuse delamination artefacts for interface 'i0': diffuse_raw (5 frames), diffuse (5 frames), combined (5 frames), metrics (5 rows).


'sample-1-quickstart'

## 5. Building the specimen the easy way

`Specimen.from_cross_ply` is convenience sugar over the three calls used in step 1: it builds a `Specimen` and automatically adds two plies at `[0, 90]` and one `"0/90"` interface between them, for the common `[0, 90]` cross-ply case. Below it's used to build the same specimen as step 1, followed by the same crack + delamination workflow to confirm it behaves identically.

In [9]:
specimen_quickstart = Specimen.from_cross_ply(
    name="sample-1-quickstart-shortcut",
    scale_px_mm=41.03328366,
    path_full=str(data_root / "full"),
    path_upper_border=str(data_root / "upper"),
    path_middle=str(data_root / "middle"),
    path_lower_border=str(data_root / "lower"),
    image_types=["png"],
    results_root=str(repo_root / "results"),
    avg_crack_width_px=8.0,
)

# from_cross_ply already added a single "0/90" interface between the two plies
interface_quickstart = specimen_quickstart.interfaces[0]

[ply.name for ply in specimen_quickstart.plies], interface_quickstart.name

(['ply_0', 'ply_1'], '0/90')

In [10]:
crack_results_quickstart = crack_analysis(
    specimen_quickstart,
    export_images=True,
    background=True,
    save_cracks=True,
)

crack_results_quickstart.keys()

dict_keys(['0', '90'])

In [11]:
detector_quickstart = DelaminationDetector(specimen_quickstart, interface_quickstart, save_preprocess_outputs=True)

result_quickstart = detector_quickstart.detect_both_delaminations(
    cracks=crack_results_quickstart,
    avg_crack_width_px=8.0,
    save_overlays=True,
    overlay_view="classified",
    save_component_overlays=True,
    save_masks=True,
    save_metrics=True,
    edge_exclusion_px=5,
    progress=True,
)

result_quickstart["paths"]

[progress] preprocess_stack: start (5 frames)


[progress] preprocess_stack: 25% (2/5)


[progress] preprocess_stack: 50% (3/5)


[progress] preprocess_stack: 75% (4/5)


[progress] preprocess_stack: 90% (5/5)


...

[progress] combined_delamination: 50% (3/5)


[progress] combined_delamination: 75% (4/5)


[progress] combined_delamination: 90% (5/5)


[progress] combined_delamination: done (5/5)


{'edge_raw_masks': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\\results\\sample-1-quickstart-shortcut\\delamination\\both\\masks\\edge_raw.npz',
 'edge_exclusion_masks': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\\results\\sample-1-quickstart-shortcut\\delamination\\both\\masks\\edge_exclusion.npz',
 'diffuse_raw_masks': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\\results\\sample-1-quickstart-shortcut\\delamination\\both\\masks\\diffuse_raw.npz',
 'diffuse_masks': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\\results\\sample-1-quickstart-shortcut\\delamination\\both\\masks\\diffuse_final.npz',
 'combined_masks': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\\results\\sample-1-quickstart-shortcut\\delamination\\both\\masks\\combined.npz',
 'metrics': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\\results\\sample-1-quickstart-shortcut\\delamination\\both\\metrics\\frame_metrics.csv',
 'combined_overlays': 'C:\\Users\\p2321038\\Documents\\GitHub\\DelaDect\